# Data Reading and Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# membaca csv file dan disimpan dalam data
data = pd.read_csv(
    "https://raw.githubusercontent.com/SulthanAbiyyu/mini-bootcamp-bcc-2023/master/housing.csv",
)
# print data
data

In [ ]:
# ukuran data
data.shape

In [ ]:
# info data
data.info()

In [ ]:
# describe data
data.describe()

In [ ]:
# banyak data yg NaN
data.isna().sum()

In [ ]:
# banyak data yg duplikat
data.duplicated().sum()

In [ ]:
# Menampilkan baris data dengan kolom total_bedrooms yang bernilai NaN/null
data[data['total_bedrooms'].isna()]

In [ ]:
# Menampilkan baris data dengan kolom households lebih dari population
data[data['households'] > data['population']]

In [ ]:
# Menampilkan baris data dengan kolom total_bedrooms lebih dari total_rooms
data2 = data[data['total_bedrooms'] > data['total_rooms']]
data2

# Data Cleaning and Preprocessing

In [ ]:
# 1. Mengubah data NaN ke mean satu kolom
# mean_total_col = data['total_bedrooms'].mean()
# data['total_bedrooms'].fillna(value=mean_total_col, inplace=True)

In [ ]:
# 2. Mengubah data NaN ke median satu kolom
median_total_col = data['total_bedrooms'].median()
data['total_bedrooms'].fillna(value=median_total_col, inplace=True)

In [ ]:
# 3. Menghapus semua data NaN di satu kolom
# data.dropna(inplace=True)
# # Jumlah data NaN/null setelah dropna()
# data.isna().sum()

In [ ]:
# 4. Mengubah data NaN ke modus satu kolom
# mode_total_col = float(data['total_bedrooms'].mode())
# data['total_bedrooms'].fillna(value=mode_total_col, inplace=True)
# # data.isna().sum()

Remove outlier data

In [ ]:
def remove_outliers_iqr(data, column_name):
    Q1 = np.percentile(data[column_name], 25)
    Q3 = np.percentile(data[column_name], 75)
    IQR = Q3 - Q1

    maximum = Q3 + (1.5 * IQR)
    minimum = Q1 - (1.5 * IQR)

    outlier_filter = (data[column_name] < minimum) | (
        data[column_name] > maximum)

    data = data[~outlier_filter]

    return data

for x in range(4):
  data = remove_outliers_iqr(data=data, column_name="housing_median_age")
  data = remove_outliers_iqr(data=data, column_name="total_rooms")
  data = remove_outliers_iqr(data=data, column_name="total_bedrooms")
  data = remove_outliers_iqr(data=data, column_name="population")
  data = remove_outliers_iqr(data=data, column_name="households")
  data = remove_outliers_iqr(data=data, column_name="median_income")
  data = remove_outliers_iqr(data=data, column_name="median_house_value")

Clustering data

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering

x = data[['housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']].copy()
km = KMeans(n_clusters=4)
data["cluster"] = km.fit_predict(x)

# Modeling (Training)

In [ ]:
from sklearn.model_selection import train_test_split

# buat data untuk data train dan data test
train, test = train_test_split(data, test_size=0.242, random_state=2023)

print("Banyak data: ", len(data))
print("Banyak data train: ", len(train))
print("Banyak data test: ", len(test))

In [ ]:
data.columns

In [ ]:
label_regresi = "median_house_value"
label_klasifikasi = "ocean_proximity"

In [ ]:
# membuang kolom label dari data train
train_data = train.drop([label_regresi, label_klasifikasi], axis=1)

# kolom khusus labelnya aja
train_label_regresi = train[label_regresi]
train_label_klasifikasi = train[label_klasifikasi]

In [ ]:
# membuang kolom label dari data train
test_data = test.drop([label_regresi, label_klasifikasi], axis=1)

# kolom khusus labelnya aja
test_label_regresi = test[label_regresi]
test_label_klasifikasi = test[label_klasifikasi]

In [ ]:
kolom_numerik = train_data.select_dtypes(include=np.number).columns.tolist()
kolom_numerik

Training Testing data StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

# membuat objek StandardScaler
scaler = StandardScaler()
# copy data train
train_ss = train_data.copy()
# fit data train
train_ss[kolom_numerik] = scaler.fit_transform(train[kolom_numerik])
train_ss

In [ ]:
# cek standar deviasi dan mean
train_ss.describe()

In [ ]:
test_ss = test_data.copy()
# transform data test
test_ss[kolom_numerik] = scaler.transform(test[kolom_numerik])
test_ss.head()

Training Testing data MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# membuat objek MinMaxScaler
minmax = MinMaxScaler()
# copy data train
train_mm = train_data.copy()
# fit data train
train_mm[kolom_numerik] = minmax.fit_transform(train[kolom_numerik])
train_mm

In [ ]:
# cek min dan max
train_mm.describe()

In [ ]:
# misal kita pilih  minmax aja
test_mm = test_data.copy()
# transform data test
test_mm[kolom_numerik] = minmax.transform(test[kolom_numerik])
test_mm.head()

Preprocessing data Training StandardScaler

In [ ]:
# Ambil kolom yang bertipe object
kolom_objek = train_mm.select_dtypes(include=object).columns.tolist()
kolom_objek

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder()
# ubah ke bentuk ohe untuk semua kolom objek
train_ohe = ohe.fit_transform(train_ss[kolom_objek]).toarray()
# ubah ke bentuk dataframe
train_ohe = pd.DataFrame(train_ohe, columns=ohe.get_feature_names_out())
# digabung dengan data train
train_ss = pd.concat([train_ss.reset_index(drop=True), train_ohe], axis=1)
# buang kolom objek karena sudah diubah ke ohe
train_ss = train_ss.drop(kolom_objek, axis=1)

train_ss.head()

In [ ]:
test_ohe = ohe.transform(test_ss[kolom_objek]).toarray()
test_ohe = pd.DataFrame(test_ohe, columns=ohe.get_feature_names_out())
test_ss = test_ss.drop(kolom_objek, axis=1)
test_ss = pd.concat([test_ss.reset_index(drop=True), test_ohe], axis=1)
test_ss.head()

In [ ]:
# cek info dari data train, udah gaada kolom objek
test_ss.info()

Preprocessing data Training MinMaxScaler

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder()
# ubah ke bentuk ohe untuk semua kolom objek
train_ohe = ohe.fit_transform(train_mm[kolom_objek]).toarray()
# ubah ke bentuk dataframe
train_ohe = pd.DataFrame(train_ohe, columns=ohe.get_feature_names_out())
# digabung dengan data train
train_mm = pd.concat([train_mm.reset_index(drop=True), train_ohe], axis=1)
# buang kolom objek karena sudah diubah ke ohe
train_mm = train_mm.drop(kolom_objek, axis=1)

train_mm.head()

In [ ]:
test_ohe = ohe.transform(test_mm[kolom_objek]).toarray()
test_ohe = pd.DataFrame(test_ohe, columns=ohe.get_feature_names_out())
test_mm = test_mm.drop(kolom_objek, axis=1)
test_mm = pd.concat([test_mm.reset_index(drop=True), test_ohe], axis=1)
test_mm.head()

In [ ]:
test_mm.info()

# Evaluation (Testing)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import ElasticNet, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# buat model
# model_regresi = LinearRegression()
# model_regresi = Ridge()
# model_regresi = ElasticNet()
# model_regresi = Lasso()
model_regresi = make_pipeline(PolynomialFeatures(degree=3), LinearRegression())
# model_regresi = RandomForestClassifier()
# model_regresi = GradientBoostingClassifier()

# training model
model_regresi.fit(train_mm, train_label_regresi)

In [ ]:
# prediksi
prediksi = model_regresi.predict(test_mm)
prediksi[:5]

In [ ]:
# lihat kunci jawaban
test_label_regresi[:5]

In [ ]:
from sklearn.metrics import mean_squared_error

# skor
mse = mean_squared_error(test_label_regresi, prediksi)
rmse = np.sqrt(mse)

print("RMSE: ", rmse)